In [1]:
import os, glob
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 100)

RAW = os.path.join("..", "data", "raw", "algeria")
OUT = os.path.join("..", "data", "processed")
os.makedirs(OUT, exist_ok=True)
BOX = (18.9, 37.2, -8.8, 12.0)                           # lat_min, lat_max, lon_min, lon_max
START = {"modis": "2003-01-01", "viirs": "2012-07-04"}   # 2000-2002 dropped (Aqua not yet flying)
CELL = 0.01                                              # cell-day unit, about 1 km

In [2]:
def read(sensor):
    files = sorted(f for f in glob.glob(os.path.join(RAW, "*.csv")) if sensor in os.path.basename(f).lower())
    if not files: raise SystemExit(f"no {sensor} csv in {RAW}")
    print(f"{sensor}: files = {[os.path.basename(f) for f in files]}")
    df = pd.concat([pd.read_csv(f, low_memory=False).assign(file=os.path.basename(f)) for f in files], ignore_index=True)
    df.columns = [c.lower() for c in df.columns]
    return df

In [3]:
def run(sensor):
    df = read(sensor)
    log = [("raw rows", len(df))]
    df = df.drop_duplicates(subset=[c for c in df.columns if c != "file"])
    log.append(("after exact-duplicate removal (archive/NRT overlap)", len(df)))
    df["date"] = pd.to_datetime(df["acq_date"], errors="coerce")
    df = df[df["date"].notna()];                                      log.append(("valid date", len(df)))
    la0, la1, lo0, lo1 = BOX
    df = df[df.latitude.between(la0, la1) & df.longitude.between(lo0, lo1)]; log.append(("inside Algeria box", len(df)))
    df = df[df["date"] >= START[sensor]];                             log.append((f"date >= {START[sensor]}", len(df)))
    if sensor == "modis":
        df = df[pd.to_numeric(df["confidence"], errors="coerce") >= 30]; log.append(("confidence >= 30", len(df)))
    else:
        df = df[df["confidence"].astype(str).str.lower().isin(["n", "h", "nominal", "high"])]; log.append(("confidence n/h", len(df)))
    df["type"] = pd.to_numeric(df["type"], errors="coerce") if "type" in df else np.nan   # NaN = no label
    log.append(("  of which type 0 (vegetation)", int((df.type == 0).sum())))
    log.append(("  of which type 2/3 (static / offshore)", int(df.type.isin([2, 3]).sum())))
    log.append(("  of which type missing", int(df.type.isna().sum())))
    steps = pd.DataFrame(log, columns=["step", "rows"])
    steps["kept vs previous main step (%)"] = np.nan
    main = [i for i, s in enumerate(steps.step) if not s.startswith("  ")]
    for a, b in zip(main[:-1], main[1:]):
        steps.loc[b, "kept vs previous main step (%)"] = round(100 * steps.rows[b] / max(steps.rows[a], 1), 1)

    df["year"] = df["date"].dt.year
    df["cell_lat"] = np.floor(df.latitude / CELL).astype(int); df["cell_lon"] = np.floor(df.longitude / CELL).astype(int)
    df["cell_day"] = df.cell_lat.astype(str) + "_" + df.cell_lon.astype(str) + "_" + df["date"].dt.strftime("%Y%m%d")
    g = df.groupby("year")
    y = pd.DataFrame({"rows": g.size(), "cell_days": g["cell_day"].nunique(),
                      "type0 %": g["type"].apply(lambda s: round(100 * (s == 0).mean(), 1)),
                      "static %": g["type"].apply(lambda s: round(100 * s.isin([2, 3]).mean(), 1)),
                      "no label %": g["type"].apply(lambda s: round(100 * s.isna().mean(), 1))})
    if sensor == "modis" and "satellite" in df:
        sat = df.groupby(["year", df["satellite"].astype(str).str[0].str.upper()]).size().unstack(fill_value=0)
        sat.columns = [{"T": "Terra rows", "A": "Aqua rows"}.get(c, c) for c in sat.columns]
        y = y.join(sat)
    m = df.groupby(df["date"].dt.to_period("M")).agg(rows=("date", "size"), days_with_data=("date", "nunique"))
    steps.to_csv(os.path.join(OUT, f"step1_{sensor}_filter_log.csv"), index=False)
    y.to_csv(os.path.join(OUT, f"step1_{sensor}_per_year.csv"))
    m.to_csv(os.path.join(OUT, f"step1_{sensor}_per_month.csv"))
    keep = df.drop(columns=["cell_day"])
    try: keep.to_parquet(os.path.join(OUT, f"{sensor}_step1.parquet"))
    except Exception: keep.to_csv(os.path.join(OUT, f"{sensor}_step1.csv"), index=False)
    return df, steps, y, m

In [4]:
modis, m_steps, m_year, m_month = run("modis")
print("\n===== MODIS: rows at each step =====");  print(m_steps.to_string(index=False))

modis: files = ['modis_archive.csv', 'modis_nrt.csv']

===== MODIS: rows at each step =====
                                               step   rows  kept vs previous main step (%)
                                           raw rows 232363                             NaN
after exact-duplicate removal (archive/NRT overlap) 232363                           100.0
                                         valid date 232363                           100.0
                                 inside Algeria box 232363                           100.0
                                 date >= 2003-01-01 221569                            95.4
                                   confidence >= 30 212300                            95.8
                       of which type 0 (vegetation)  80164                             NaN
              of which type 2/3 (static / offshore) 131226                             NaN
                              of which type missing    910                             Na

In [5]:
print("===== MODIS: per year (after the filters above) =====");  print(m_year.to_string())

===== MODIS: per year (after the filters above) =====
       rows  cell_days  type0 %  static %  no label %  Aqua rows  Terra rows
year                                                                        
2003   8749       7732     30.0      70.0         0.0       4337        4412
2004   8385       7505     39.2      60.8         0.0       4127        4258
2005   8832       7872     40.9      59.1         0.0       4297        4535
2006  10449       9184     33.2      66.8         0.0       5145        5304
2007  12640      10966     42.1      57.9         0.0       6667        5973
2008  10477       9236     27.1      72.9         0.0       5134        5343
2009   9939       8697     37.6      62.4         0.0       4951        4988
2010  10118       8980     32.5      67.5         0.0       5030        5088
2011   8308       7344     29.7      70.3         0.0       4024        4284
2012  15265      13382     60.0      40.0         0.0       8152        7113
2013   9189       7949

In [7]:
viirs, v_steps, v_year, v_month = run("viirs")
print("\n===== VIIRS: rows at each step =====");  print(v_steps.to_string(index=False))

viirs: files = ['viirs_archive.csv', 'viirs_nrt.csv']

===== VIIRS: rows at each step =====
                                               step    rows  kept vs previous main step (%)
                                           raw rows 1402416                             NaN
after exact-duplicate removal (archive/NRT overlap) 1402416                           100.0
                                         valid date 1402416                           100.0
                                 inside Algeria box 1402416                           100.0
                                 date >= 2012-07-04 1402416                           100.0
                                     confidence n/h 1310267                            93.4
                       of which type 0 (vegetation)  252649                             NaN
              of which type 2/3 (static / offshore) 1031600                             NaN
                              of which type missing   26018                     

In [8]:
print("===== VIIRS: per year (after the filters above) =====");  print(v_year.to_string())

===== VIIRS: per year (after the filters above) =====
        rows  cell_days  type0 %  static %  no label %
year                                                  
2012   63896      32523     43.3      56.7         0.0
2013   85812      44625     16.5      83.5         0.0
2014   96190      50440     24.2      75.8         0.0
2015   89411      48013     17.1      82.9         0.0
2016   94646      51213     21.2      78.8         0.0
2017  100648      54661     23.3      76.7         0.0
2018   89597      49231     15.4      84.6         0.0
2019   96899      53079     20.4      79.6         0.0
2020   95462      51565     19.8      80.2         0.0
2021   83377      45509     17.5      82.5         0.0
2022   85803      48027     17.7      82.3         0.0
2023   88857      50555     17.6      82.4         0.0
2024   80827      46208     15.0      85.0         0.0
2025   87589      49340     11.1      88.9         0.0
2026   71253      39984     12.5      51.0        36.5


In [9]:
for s in ("modis", "viirs"):
    raw = read(s); raw["date"] = pd.to_datetime(raw["acq_date"])
    print(f"\n{s.upper()} date range per file"); print(raw.groupby("file")["date"].agg(["min", "max", "size"]))
    pm = raw.groupby([raw["date"].dt.to_period("M"), "file"]).size().unstack(fill_value=0)
    print(f"{s.upper()} rows per month per file, last 14 months"); print(pm.tail(14))

modis: files = ['modis_archive.csv', 'modis_nrt.csv']

MODIS date range per file
                         min        max    size
file                                           
modis_archive.csv 2000-11-01 2026-08-31  231403
modis_nrt.csv     2026-09-01 2026-11-06     960
MODIS rows per month per file, last 14 months
file     modis_archive.csv  modis_nrt.csv
date                                     
2025-10                588              0
2025-11                258              0
2025-12                207              0
2026-01                139              0
2026-02                193              0
2026-03                184              0
2026-04                288              0
2026-05                367              0
2026-06                526              0
2026-07               1864              0
2026-08               2376              0
2026-09                  0            550
2026-10                  0            360
2026-11                  0             50
viirs: fi

In [10]:
for name, d in (("MODIS", modis), ("VIIRS", viirs)):
    t = d.assign(t=d["type"].fillna(-1).astype(int)).groupby(["year", "t"]).size().unstack(fill_value=0)
    print(f"\n{name} rows by type (-1 = no label)"); print(t.to_string())

raw = read("modis"); raw["year"] = pd.to_datetime(raw["acq_date"]).dt.year
print("\nMODIS raw rows by satellite, 2000-2003")
print(raw[raw.year <= 2003].groupby(["year", "satellite"]).size().unstack(fill_value=0))


MODIS rows by type (-1 = no label)
t      -1     0     2   3
year                     
2003    0  2629  6120   0
2004    0  3286  5099   0
2005    0  3608  5224   0
2006    0  3468  6980   1
2007    0  5317  7321   2
2008    0  2839  7636   2
2009    0  3741  6197   1
2010    0  3285  6833   0
2011    0  2466  5842   0
2012    0  9159  6105   1
2013    0  1970  7219   0
2014    0  4368  6401  72
2015    0  2048  6719  32
2016    0  3034  6679  16
2017    0  4478  5683   7
2018    0  1241  5226   6
2019    0  3722  4963   4
2020    0  4607  6486  10
2021    0  3301  4177   1
2022    0  2535  3160   2
2023    0  2255  2786  10
2024    0  1456  2877   2
2025    0  1476  3450   2
2026  910  3875  1868   4

VIIRS rows by type (-1 = no label)
t        -1      0      2     3
year                           
2012      0  27640  36161    95
2013      0  14197  71282   333
2014      0  23256  71513  1421
2015      0  15321  73130   960
2016      0  20076  73734   836
2017      0  23480  76717   

In [11]:
print("MODIS"); print(m_month.sort_values("days_with_data").head(20))
print("VIIRS"); print(v_month.sort_values("days_with_data").head(20))

MODIS
         rows  days_with_data
date                         
2026-11    47               6
2024-02   101              23
2024-01   120              24
2018-02   176              25
2026-01   127              25
2014-02   263              25
2007-02   303              26
2022-04   230              26
2017-02   319              26
2021-02   405              27
2023-02   207              27
2015-02   312              27
2013-02   323              27
2019-02   300              27
2022-02   158              27
2003-02   335              27
2018-11   298              28
2018-10   436              28
2022-11   314              28
2022-01   157              28
VIIRS
          rows  days_with_data
date                          
2026-11    569               2
2024-07   4660              19
2022-08   6805              21
2026-07  10090              25
2024-11   7299              26
2022-07   6504              26
2026-04   6477              27
2024-06   5934              27
2019-02   6455    

In [12]:
OLD = ["2017-12","2018-10","2022-04","2022-07","2022-08","2024-01","2024-02","2024-06","2024-07","2024-11","2025-12","2026-01","2026-03","2026-04","2026-07"]

def gap_table(dates, start, end, name):
    all_days = pd.date_range(start, end)
    have = pd.DatetimeIndex(pd.Series(dates).dt.normalize().unique())
    miss = all_days.difference(have)
    if len(miss) == 0:
        return pd.DataFrame(columns=[f"{name}_missing", f"{name}_longest_gap"])
    s = pd.Series(miss)
    run_id = (s.diff().dt.days != 1).cumsum().values
    run_len = s.groupby(run_id).transform("size").values
    per = miss.to_period("M")
    out = pd.DataFrame({"m": per, "run": run_len}).groupby("m").agg(**{f"{name}_missing": ("run", "size"), f"{name}_longest_gap": ("run", "max")})
    return out

sat = modis["satellite"].astype(str).str[0].str.upper()
parts = [
    gap_table(viirs["date"], "2012-07-04", viirs["date"].max(), "viirs"),
    gap_table(modis.loc[sat == "T", "date"], "2003-01-01", modis["date"].max(), "terra"),
    gap_table(modis.loc[sat == "A", "date"], "2003-01-01", modis["date"].max(), "aqua"),
]
gaps = pd.concat(parts, axis=1).fillna(0).astype(int)
gaps.index = gaps.index.astype(str)
gaps["worst"] = gaps[[c for c in gaps.columns if c.endswith("_missing")]].max(axis=1)

print("The 15 months you excluded before: missing days (and longest run of missing days) per sensor")
print(gaps.reindex(OLD).fillna(0).astype(int).to_string())
print("\nOther months with 3 or more missing days for some sensor (not in the old list)")
print(gaps[(gaps.worst >= 3) & (~gaps.index.isin(OLD))].sort_values("worst", ascending=False).head(25).to_string())

The 15 months you excluded before: missing days (and longest run of missing days) per sensor
         viirs_missing  viirs_longest_gap  terra_missing  terra_longest_gap  aqua_missing  aqua_longest_gap  worst
m                                                                                                                 
2017-12              0                  0              6                  1             6                 2      6
2018-10              0                  0              7                  1             6                 3      7
2022-04              0                  0              6                  1            19                16     19
2022-07              5                 15              0                  0             1                 1      5
2022-08             10                 15              0                  0             0                 0     10
2024-01              1                  1             11                  1            15             

In [13]:
OLD = ["2017-12","2018-10","2022-04","2022-07","2022-08","2024-01","2024-02","2024-06","2024-07","2024-11","2025-12","2026-01","2026-03","2026-04","2026-07"]
MINRUN = {"viirs": 1, "terra": 3, "aqua": 3}   # shortest run of empty days that counts as lost data
THRESH = 0.10                                  # flag a sensor-month if >= 10% of its days are lost

def lost_days(dates, start, end, minrun, name):
    all_days = pd.date_range(start, end)
    have = pd.DatetimeIndex(pd.Series(dates).dt.normalize().unique())
    miss = all_days.difference(have)
    n_days = pd.Series(1, index=all_days).groupby(all_days.to_period("M")).sum()
    lost = pd.Series(0, index=n_days.index)
    if len(miss):
        s = pd.Series(miss)
        run_id = (s.diff().dt.days != 1).cumsum().values
        run_len = s.groupby(run_id).transform("size").values
        kept = miss[run_len >= minrun]
        if len(kept):
            lost = pd.Series(1, index=kept).groupby(kept.to_period("M")).sum()
    out = pd.DataFrame({"days": n_days}).join(lost.rename("lost")).fillna(0)
    out[f"{name}_lost"] = out["lost"].astype(int)
    out[f"{name}_frac"] = out["lost"] / out["days"]
    return out[[f"{name}_lost", f"{name}_frac"]]

sat = modis["satellite"].astype(str).str[0].str.upper()
L = pd.concat([
    lost_days(viirs["date"], "2012-07-04", viirs["date"].max(), MINRUN["viirs"], "viirs"),
    lost_days(modis.loc[sat == "T", "date"], "2003-01-01", modis["date"].max(), MINRUN["terra"], "terra"),
    lost_days(modis.loc[sat == "A", "date"], "2003-01-01", modis["date"].max(), MINRUN["aqua"], "aqua"),
], axis=1).fillna(0)
L.index = L.index.astype(str)
for c in ("viirs_lost", "terra_lost", "aqua_lost"): L[c] = L[c].astype(int)
f = pd.DataFrame({"viirs": L.viirs_frac >= THRESH, "terra": L.terra_frac >= THRESH, "aqua": L.aqua_frac >= THRESH})
L["flagged_by"] = f.apply(lambda r: ",".join(k for k in r.index if r[k]), axis=1)
flagged = L[L.flagged_by != ""]

print(f"Months flagged (lost days >= {THRESH:.0%} of the month): {len(flagged)}")
print(flagged[["viirs_lost", "terra_lost", "aqua_lost", "flagged_by"]].to_string())
new, old = set(flagged.index), set(OLD)
print("\nOLD list and flagged :", sorted(old & new))
print("OLD list, NOT flagged:", sorted(old - new))
print("Flagged, NOT in OLD   :", sorted(new - old))
print("\nLost days for the old months that are NOT flagged:")
print(L.reindex(sorted(old - new))[["viirs_lost", "terra_lost", "aqua_lost"]].to_string())

Months flagged (lost days >= 10% of the month): 19
         viirs_lost  terra_lost  aqua_lost flagged_by
2015-02           0           0          3       aqua
2016-02           0           9          0      terra
2017-06           0           3          0      terra
2018-02           0           0          3       aqua
2021-11           0           0          3       aqua
2022-01           0           4          3      terra
2022-04           0           0         16       aqua
2022-07           5           0          0      viirs
2022-08          10           0          0      viirs
2022-10           0          14          0      terra
2024-02           0           0          3       aqua
2024-06           3           0          0      viirs
2024-07          12           0          0      viirs
2024-11           4           0          0      viirs
2026-01           0           0          8       aqua
2026-04           3           0          0      viirs
2026-07           6           0

In [14]:
OLD = ["2017-12","2018-10","2022-04","2022-07","2022-08","2024-01","2024-02","2024-06","2024-07","2024-11","2025-12","2026-01","2026-03","2026-04","2026-07"]
MINRUN = {"viirs": 1, "terra": 3, "aqua": 3}   # shortest run of empty days that counts as lost data
THRESH = 0.10                                  # flag a sensor-month if >= 10% of its days are lost

def lost_days(dates, start, end, minrun, name):
    all_days = pd.date_range(start, end)
    have = pd.DatetimeIndex(pd.Series(dates).dt.normalize().unique())
    miss = all_days.difference(have)
    n_days = pd.Series(1, index=all_days).groupby(all_days.to_period("M")).sum()
    lost = pd.Series(0, index=n_days.index)
    if len(miss):
        s = pd.Series(miss)
        run_id = (s.diff().dt.days != 1).cumsum().values
        run_len = s.groupby(run_id).transform("size").values
        kept = miss[run_len >= minrun]
        if len(kept):
            lost = pd.Series(1, index=kept).groupby(kept.to_period("M")).sum()
    out = pd.DataFrame({"days": n_days}).join(lost.rename("lost")).fillna(0)
    out[f"{name}_lost"] = out["lost"].astype(int)
    out[f"{name}_frac"] = out["lost"] / out["days"]
    return out[[f"{name}_lost", f"{name}_frac"]]

sat = modis["satellite"].astype(str).str[0].str.upper()
L = pd.concat([
    lost_days(viirs["date"], "2012-07-04", viirs["date"].max(), MINRUN["viirs"], "viirs"),
    lost_days(modis.loc[sat == "T", "date"], "2003-01-01", modis["date"].max(), MINRUN["terra"], "terra"),
    lost_days(modis.loc[sat == "A", "date"], "2003-01-01", modis["date"].max(), MINRUN["aqua"], "aqua"),
], axis=1).fillna(0)
L.index = L.index.astype(str)
for c in ("viirs_lost", "terra_lost", "aqua_lost"): L[c] = L[c].astype(int)
f = pd.DataFrame({"viirs": L.viirs_frac >= THRESH, "terra": L.terra_frac >= THRESH, "aqua": L.aqua_frac >= THRESH})
L["flagged_by"] = f.apply(lambda r: ",".join(k for k in r.index if r[k]), axis=1)
flagged = L[L.flagged_by != ""]

print(f"Months flagged (lost days >= {THRESH:.0%} of the month): {len(flagged)}")
print(flagged[["viirs_lost", "terra_lost", "aqua_lost", "flagged_by"]].to_string())
new, old = set(flagged.index), set(OLD)
print("\nOLD list and flagged :", sorted(old & new))
print("OLD list, NOT flagged:", sorted(old - new))
print("Flagged, NOT in OLD   :", sorted(new - old))
print("\nLost days for the old months that are NOT flagged:")
print(L.reindex(sorted(old - new))[["viirs_lost", "terra_lost", "aqua_lost"]].to_string())

Months flagged (lost days >= 10% of the month): 19
         viirs_lost  terra_lost  aqua_lost flagged_by
2015-02           0           0          3       aqua
2016-02           0           9          0      terra
2017-06           0           3          0      terra
2018-02           0           0          3       aqua
2021-11           0           0          3       aqua
2022-01           0           4          3      terra
2022-04           0           0         16       aqua
2022-07           5           0          0      viirs
2022-08          10           0          0      viirs
2022-10           0          14          0      terra
2024-02           0           0          3       aqua
2024-06           3           0          0      viirs
2024-07          12           0          0      viirs
2024-11           4           0          0      viirs
2026-01           0           0          8       aqua
2026-04           3           0          0      viirs
2026-07           6           0